# Document Q/A Rag System

In [ ]:
from google.colab import userdata
import os
os.environ['GOOGLE_API_KEY'] = userdata.get("GOOGLE_API_KEY")
os.environ['HUGGINGFACEHUB_ACCESS_TOKEN'] = userdata.get("HUGGINGFACEHUB_ACCESS_TOKEN")

In [ ]:
!pip -q install langchain langchain-google-genai langchain-community faiss-cpu tiktoken python-dotenv pypdf langchain-huggingface

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI,GoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings

# Testing

In [ ]:
chat_model = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
chat_model.invoke("HI")

# Step 1a - Indexing (Document Ingestion)

In [ ]:
loader = PyPDFLoader("/content/Docker Deep Dive.pdf")
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
docs[0]

# Step 1b - Indexing(Text Splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(docs)

In [ ]:
len(chunks)

In [ ]:
chunks[200]

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

# Step 2 - Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke("What is docker")

# Step 3 - Augmentation

In [ ]:
llm = GoogleGenerativeAI(model="models/gemini-1.5-flash")

In [ ]:
prompt = PromptTemplate(
    template = """
    You are a helpful assistant.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}

    Question: {question}
    """,
    input_variables=["context","question"]
)

In [ ]:
question = "if the topic of aliens disscussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
context_text

In [ ]:
final_prompt = prompt.invoke({"context":context_text,"question":question})

In [ ]:
final_prompt

# Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)

In [ ]:
answer

# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
}
)

In [ ]:
parallel_chain.invoke('How to manage a app in a container')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke("How to manage a app in a container")

In [ ]:
main_chain.invoke("How to manage a app in a container. Guide me in detail")